<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/python/notebooks/c2_l4.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C2-L4 · Gráficos y underwater
Precio, equity y drawdown con matplotlib y plotly. El underwater muestra cuánto dolió ganar.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/python/data/c2_l4.csv'
try:
    df = pd.read_csv(URL, parse_dates=['date'])
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c2_l4.csv'), Path('data/c2_l4.csv'), Path('c2_l4.csv')]:
        if cand.exists():
            df = pd.read_csv(cand, parse_dates=['date']); break
    print('Fuente: local')
df = df.sort_values('date').reset_index(drop=True)
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(df['date'], df['close']); ax[0].set_title('Precio')
ax[1].plot(df['date'], df['equity'], color='green'); ax[1].set_title('Equity')
plt.tight_layout(); plt.savefig('c2_l4_equity.png')
print('guardado: c2_l4_equity.png')

In [ ]:
df['peak'] = df['equity'].cummax()
df['drawdown'] = (df['equity'] - df['peak']) / df['peak']
max_dd = df['drawdown'].min()
fecha_dd = df.loc[df['drawdown'].idxmin(), 'date']
print(f'max_drawdown={max_dd:.2%}  fecha={fecha_dd.date()}')
recup = df.loc[df['date'] > fecha_dd]
vuelta = recup[recup['equity'] >= df['peak'].max()]
print('velas_hasta_recuperar:', len(vuelta) and int(vuelta.index[0] - df['drawdown'].idxmin()) or 'sin recuperar en la muestra')
try:
    import plotly.graph_objects as go
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df['date'], y=df['equity'], name='equity'))
    fig.add_trace(go.Scatter(x=df['date'], y=df['drawdown'], name='underwater', fill='tozeroy'))
    fig.write_html('c2_l4_underwater.html')
    print('guardado: c2_l4_underwater.html')
except ImportError:
    print('plotly no instalado: underwater solo en matplotlib')

In [ ]:
# Chequeo automático
assert (df['drawdown'] <= 1e-12).all(), 'drawdown nunca positivo'
assert abs(df['peak'].iloc[-1] - df['equity'].max()) < 1e-9, 'pico final = máximo del equity'
assert df['drawdown'].min() < 0, 'debe existir al menos una caída'
assert df['peak'].iloc[0] == df['equity'].iloc[0]
print(f'OK: underwater verificado, max_dd={df["drawdown"].min():.2%}')